# Parameter Tuning (Two-Stage K-Means)

Purpose: evaluate the core parameter grids using only the **21-image tuning split** and automatically carry forward the highest-performing configuration.

Selection rule used in this notebook:

- **Banana mask:** highest mean Dice, then highest mean IoU as tie-breaker.
- **Blemish mask:** highest mean Dice, then highest mean IoU, then lower blemish-percentage MAE as tie-breaker.

The final classical pipeline uses two K-Means stages: whole-image K-Means for banana segmentation and second-stage Lab a,b K-Means for blemish detection. No supplemental Rule A/B tuning is performed.


In [1]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "banana_quality" / "config.py").exists()
            and (candidate / "data" / "annotations").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from inside the banana project."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.segmentation.kmeans import prepare_image
from banana_quality.segmentation.mask_refinement import banana_mask_from_prepared
from banana_quality.blemish.detection import blemish_mask_from_prepared
from banana_quality.surface_quality import blemish_percentage

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)


Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system
Dataset root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\data\annotations


In [2]:
images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

split_df = pd.DataFrame([
    {
        "image": p.name,
        "split": "final_evaluation" if p.name in final_names else "tuning",
    }
    for p in image_paths
]).sort_values(["split", "image"]).reset_index(drop=True)

split_df.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert int((split_df["split"] == "tuning").sum()) == 21
assert int((split_df["split"] == "final_evaluation").sum()) == 9

print(split_df["split"].value_counts())


split
tuning              21
final_evaluation     9
Name: count, dtype: int64


In [3]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def resize_binary_mask(mask: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    h, w = target_shape
    resized = cv2.resize(
        np.asarray(mask, dtype=np.uint8),
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )
    return np.where(resized > 0, 255, 0).astype(np.uint8)


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)


def load_samples(split_name: str):
    names = set(split_df.loc[split_df["split"] == split_name, "image"])
    samples = []
    for image_path in image_paths:
        if image_path.name not in names:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        shape = image.shape[:2]
        banana_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "banana_masks", image_path),
            shape,
        )
        blemish_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "blemish_masks", image_path),
            shape,
        )
        samples.append({
            "name": image_path.name,
            "image": image,
            "shape": shape,
            "banana_gt": banana_gt,
            "blemish_gt": blemish_gt,
        })
    return samples



tuning_samples = load_samples("tuning")
print("Loaded tuning images:", len(tuning_samples))


Loaded tuning images: 21


## 1. Banana K + cluster-acceptance grid


In [4]:
DEFAULTS = ProcessingConfig()
BANANA_K_VALUES = [4, 5, 6, 7, 8]
BANANA_ACCEPTANCE_VALUES = [0.35, 0.40, 0.45, 0.50, 0.55]
BASE_BANANA_OPENING = int(DEFAULTS.banana_opening_kernel)
BASE_BANANA_CLOSING = int(DEFAULTS.banana_closing_kernel)


def evaluate_banana(k, acceptance, opening, closing):
    config = replace(
        DEFAULTS,
        kmeans_k=int(k),
        banana_cluster_score_threshold=float(acceptance),
    )
    ious, dices = [], []
    for sample in tuning_samples:
        prepared = prepare_image(sample["image"], config)
        steps = banana_mask_from_prepared(
            prepared,
            opening_kernel_size=int(opening),
            closing_kernel_size=int(closing),
        )
        pred = resize_binary_mask(steps["banana_mask"], sample["shape"])
        ious.append(iou_score(pred, sample["banana_gt"]))
        dices.append(dice_score(pred, sample["banana_gt"]))
    return {
        "kmeans_k": int(k),
        "banana_cluster_score_threshold": float(acceptance),
        "banana_opening_kernel": int(opening),
        "banana_closing_kernel": int(closing),
        "banana_mean_iou": float(np.mean(ious)),
        "banana_mean_dice": float(np.mean(dices)),
    }

rows = []
for k, acceptance in product(BANANA_K_VALUES, BANANA_ACCEPTANCE_VALUES):
    rows.append(
        evaluate_banana(
            k,
            acceptance,
            BASE_BANANA_OPENING,
            BASE_BANANA_CLOSING,
        )
    )

banana_pair_results = (
    pd.DataFrame(rows)
    .sort_values(
        ["banana_mean_dice", "banana_mean_iou"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)
banana_pair_results.to_csv(
    RESULTS_DIR / "tuning_banana_k_acceptance.csv",
    index=False,
)
display(banana_pair_results)
print("Saved tuning_banana_k_acceptance.csv")


,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,banana_mean_iou,banana_mean_dice
0,7,0.35,3,7,0.927202,0.960584
1,6,0.35,3,7,0.922589,0.957476
2,6,0.40,3,7,0.919666,0.955899
3,7,0.40,3,7,0.913893,0.952396
4,7,0.45,3,7,0.913893,0.952396
5,5,0.35,3,7,0.904742,0.947947
6,8,0.35,3,7,0.906236,0.946683
7,8,0.40,3,7,0.906236,0.946683
8,8,0.45,3,7,0.906236,0.946683
9,6,0.45,3,7,0.904441,0.946314


Saved tuning_banana_k_acceptance.csv


### Automatically selected banana K and acceptance

The first row is carried forward because it has the highest mean Dice (with mean IoU as tie-breaker).


In [5]:
best_banana_pair = banana_pair_results.iloc[0]
SELECTED_BANANA_K = int(best_banana_pair["kmeans_k"])
SELECTED_BANANA_ACCEPTANCE = float(
    best_banana_pair["banana_cluster_score_threshold"]
)

print("Selected banana K:", SELECTED_BANANA_K)
print("Selected banana acceptance:", SELECTED_BANANA_ACCEPTANCE)
print("Mean Dice:", float(best_banana_pair["banana_mean_dice"]))
print("Mean IoU:", float(best_banana_pair["banana_mean_iou"]))


Selected banana K: 7
Selected banana acceptance: 0.35
Mean Dice: 0.9605840001931989
Mean IoU: 0.9272024356343475


## 2. Banana morphology grid


In [6]:
BANANA_OPENING_VALUES = [1, 3, 5]
BANANA_CLOSING_VALUES = [1, 3, 5, 7]
rows = []
for opening, closing in product(BANANA_OPENING_VALUES, BANANA_CLOSING_VALUES):
    rows.append(
        evaluate_banana(
            SELECTED_BANANA_K,
            SELECTED_BANANA_ACCEPTANCE,
            opening,
            closing,
        )
    )

banana_morph_results = (
    pd.DataFrame(rows)
    .sort_values(
        ["banana_mean_dice", "banana_mean_iou"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)
banana_morph_results.to_csv(
    RESULTS_DIR / "tuning_banana_morphology.csv",
    index=False,
)
display(banana_morph_results)
print("Saved tuning_banana_morphology.csv")


,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,banana_mean_iou,banana_mean_dice
0,7,0.35,3,7,0.927202,0.960584
1,7,0.35,3,5,0.925407,0.959480
2,7,0.35,5,7,0.925397,0.959420
3,7,0.35,3,3,0.923342,0.958333
4,7,0.35,5,5,0.923249,0.958136
5,7,0.35,3,1,0.922496,0.957770
6,7,0.35,5,3,0.919247,0.955732
7,7,0.35,5,1,0.917383,0.954729
8,7,0.35,1,7,0.919428,0.954637
9,7,0.35,1,3,0.917879,0.954210


Saved tuning_banana_morphology.csv


### Automatically selected banana morphology

Again, the first row is carried forward from the tuning results.


In [7]:
best_banana_morph = banana_morph_results.iloc[0]
SELECTED_BANANA_OPENING = int(best_banana_morph["banana_opening_kernel"])
SELECTED_BANANA_CLOSING = int(best_banana_morph["banana_closing_kernel"])

print("Selected banana opening:", SELECTED_BANANA_OPENING)
print("Selected banana closing:", SELECTED_BANANA_CLOSING)
print("Mean Dice:", float(best_banana_morph["banana_mean_dice"]))
print("Mean IoU:", float(best_banana_morph["banana_mean_iou"]))


Selected banana opening: 3
Selected banana closing: 7
Mean Dice: 0.9605840001931989
Mean IoU: 0.9272024356343475


## 3. Build the banana masks used for blemish tuning


In [8]:
banana_config = replace(
    DEFAULTS,
    kmeans_k=int(SELECTED_BANANA_K),
    banana_cluster_score_threshold=float(SELECTED_BANANA_ACCEPTANCE),
)

blemish_cache = []
for sample in tuning_samples:
    prepared = prepare_image(sample["image"], banana_config)
    banana_steps = banana_mask_from_prepared(
        prepared,
        opening_kernel_size=int(SELECTED_BANANA_OPENING),
        closing_kernel_size=int(SELECTED_BANANA_CLOSING),
    )
    banana_working = np.asarray(banana_steps["banana_mask"], dtype=np.uint8)
    banana_pred = resize_binary_mask(banana_working, sample["shape"])
    blemish_cache.append({
        **sample,
        "prepared": prepared,
        "banana_working": banana_working,
        "banana_pred": banana_pred,
    })
print("Cached tuning images:", len(blemish_cache))


Cached tuning images: 21


## 4. Blemish K + chroma-distance grid


In [9]:
BLEMISH_K_VALUES = [3, 4, 5, 6, 7]
BLEMISH_CHROMA_VALUES = [12.0, 15.0, 18.0, 21.0, 24.0]
BASE_BLEMISH_OPENING = int(DEFAULTS.blemish_opening_kernel)
BASE_BLEMISH_CLOSING = int(DEFAULTS.blemish_closing_kernel)


def evaluate_blemish(k, chroma, opening, closing):
    config = replace(
        banana_config,
        blemish_kmeans_k=int(k),
        blemish_cluster_chroma_threshold=float(chroma),
    )
    ious, dices, pct_errors = [], [], []
    for sample in blemish_cache:
        steps = blemish_mask_from_prepared(
            sample["prepared"],
            sample["banana_working"],
            opening_kernel_size=int(opening),
            closing_kernel_size=int(closing),
            config=config,
        )
        pred = resize_binary_mask(steps["blemish_mask"], sample["shape"])
        ious.append(iou_score(pred, sample["blemish_gt"]))
        dices.append(dice_score(pred, sample["blemish_gt"]))
        pred_pct = blemish_percentage(sample["banana_pred"], pred)
        gt_pct = blemish_percentage(sample["banana_gt"], sample["blemish_gt"])
        pct_errors.append(abs(pred_pct - gt_pct))
    return {
        "blemish_kmeans_k": int(k),
        "blemish_cluster_chroma_threshold": float(chroma),
        "blemish_opening_kernel": int(opening),
        "blemish_closing_kernel": int(closing),
        "blemish_mean_iou": float(np.mean(ious)),
        "blemish_mean_dice": float(np.mean(dices)),
        "blemish_percentage_mae": float(np.mean(pct_errors)),
    }

rows = []
for k, chroma in product(BLEMISH_K_VALUES, BLEMISH_CHROMA_VALUES):
    rows.append(
        evaluate_blemish(
            k,
            chroma,
            BASE_BLEMISH_OPENING,
            BASE_BLEMISH_CLOSING,
        )
    )

blemish_pair_results = (
    pd.DataFrame(rows)
    .sort_values(
        ["blemish_mean_dice", "blemish_mean_iou", "blemish_percentage_mae"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
blemish_pair_results.to_csv(
    RESULTS_DIR / "tuning_blemish_k_chroma.csv",
    index=False,
)
display(blemish_pair_results)
print("Saved tuning_blemish_k_chroma.csv")


,blemish_kmeans_k,blemish_cluster_chroma_threshold,blemish_opening_kernel,blemish_closing_kernel,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,6,21.0,3,3,0.322921,0.416506,7.796957
1,6,15.0,3,3,0.292291,0.406746,11.815173
2,6,12.0,3,3,0.288624,0.402503,11.744991
3,7,12.0,3,3,0.290898,0.401989,10.178190
4,5,12.0,3,3,0.287148,0.394105,10.829768
5,4,12.0,3,3,0.285600,0.389049,11.619315
6,7,24.0,3,3,0.305293,0.388215,6.436585
7,6,24.0,3,3,0.304864,0.387946,7.326248
8,7,15.0,3,3,0.283297,0.386074,9.069793
9,7,18.0,3,3,0.286567,0.379403,5.650095


Saved tuning_blemish_k_chroma.csv


### Automatically selected blemish K and chroma threshold

The first row is carried forward using mean Dice → mean IoU → lower percentage MAE.


In [10]:
best_blemish_pair = blemish_pair_results.iloc[0]
SELECTED_BLEMISH_K = int(best_blemish_pair["blemish_kmeans_k"])
SELECTED_BLEMISH_CHROMA = float(
    best_blemish_pair["blemish_cluster_chroma_threshold"]
)

print("Selected blemish K:", SELECTED_BLEMISH_K)
print("Selected chroma threshold:", SELECTED_BLEMISH_CHROMA)
print("Mean Dice:", float(best_blemish_pair["blemish_mean_dice"]))
print("Mean IoU:", float(best_blemish_pair["blemish_mean_iou"]))
print("Percentage MAE:", float(best_blemish_pair["blemish_percentage_mae"]))


Selected blemish K: 6
Selected chroma threshold: 21.0
Mean Dice: 0.41650560118775165
Mean IoU: 0.322920995803499
Percentage MAE: 7.796956973549235


## 5. Blemish morphology grid


In [11]:
BLEMISH_OPENING_VALUES = [1, 3, 5]
BLEMISH_CLOSING_VALUES = [1, 3, 5, 7]
rows = []
for opening, closing in product(BLEMISH_OPENING_VALUES, BLEMISH_CLOSING_VALUES):
    rows.append(
        evaluate_blemish(
            SELECTED_BLEMISH_K,
            SELECTED_BLEMISH_CHROMA,
            opening,
            closing,
        )
    )

blemish_morph_results = (
    pd.DataFrame(rows)
    .sort_values(
        ["blemish_mean_dice", "blemish_mean_iou", "blemish_percentage_mae"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
blemish_morph_results.to_csv(
    RESULTS_DIR / "tuning_blemish_morphology.csv",
    index=False,
)
display(blemish_morph_results)
print("Saved tuning_blemish_morphology.csv")


,blemish_kmeans_k,blemish_cluster_chroma_threshold,blemish_opening_kernel,blemish_closing_kernel,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,6,21.0,3,5,0.323408,0.417085,7.828478
1,6,21.0,1,3,0.322810,0.416818,7.848011
2,6,21.0,1,5,0.322667,0.416787,7.892460
3,6,21.0,3,7,0.322920,0.416699,7.903794
4,6,21.0,3,3,0.322921,0.416506,7.796957
5,6,21.0,3,1,0.322708,0.416262,7.791651
6,6,21.0,1,7,0.321697,0.416063,8.009911
7,6,21.0,1,1,0.322156,0.416061,7.837606
8,6,21.0,5,5,0.320161,0.412856,7.768960
9,6,21.0,5,7,0.319762,0.412589,7.809966


Saved tuning_blemish_morphology.csv


## 6. Save the automatically selected two-stage K-Means parameters

The selected configuration is saved to `recommended_core_parameters.csv` and is loaded directly by the final-evaluation notebook.


In [12]:
best_blemish_morph = blemish_morph_results.iloc[0]
SELECTED_BLEMISH_OPENING = int(best_blemish_morph["blemish_opening_kernel"])
SELECTED_BLEMISH_CLOSING = int(best_blemish_morph["blemish_closing_kernel"])

recommended_core = pd.DataFrame([{
    "kmeans_k": SELECTED_BANANA_K,
    "banana_cluster_score_threshold": SELECTED_BANANA_ACCEPTANCE,
    "banana_opening_kernel": SELECTED_BANANA_OPENING,
    "banana_closing_kernel": SELECTED_BANANA_CLOSING,
    "blemish_kmeans_k": SELECTED_BLEMISH_K,
    "blemish_cluster_chroma_threshold": SELECTED_BLEMISH_CHROMA,
    "blemish_opening_kernel": SELECTED_BLEMISH_OPENING,
    "blemish_closing_kernel": SELECTED_BLEMISH_CLOSING,
    "banana_mean_iou": float(best_banana_morph["banana_mean_iou"]),
    "banana_mean_dice": float(best_banana_morph["banana_mean_dice"]),
    "blemish_mean_iou": float(best_blemish_morph["blemish_mean_iou"]),
    "blemish_mean_dice": float(best_blemish_morph["blemish_mean_dice"]),
    "blemish_percentage_mae": float(best_blemish_morph["blemish_percentage_mae"]),
}])

recommended_core.to_csv(
    RESULTS_DIR / "recommended_core_parameters.csv",
    index=False,
)

display(recommended_core)
print("Saved recommended_core_parameters.csv")
print("Next: run final_evaluation.ipynb.")


,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,blemish_kmeans_k,blemish_cluster_chroma_threshold,blemish_opening_kernel,blemish_closing_kernel,banana_mean_iou,banana_mean_dice,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,7,0.35,3,7,6,21.0,3,5,0.927202,0.960584,0.323408,0.417085,7.828478


Saved recommended_core_parameters.csv
Next: run final_evaluation.ipynb.
